# Normal Drift Baseline

Reproduces the drift detection experiment from:
> Kuppa & Le-Khac (2022) *Learn to adapt: Robust drift detection in security domain.*

**Dataset**: CICIDS2017 (Engelen et al. corrected)  
**Protocol**: Drop novel attack classes from training; evaluate drift detection on test set.  
**Threshold**: F1-optimal sweep on calibration/validation split, evaluated on a disjoint held-out set.

| Label | Meaning |
|---|---|
| 1 (positive) | novel/drifted sample (from a dropped class) |
| 0 (negative) | known-class sample |

## 1. Imports & Configuration

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import torch
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from src.detectors.contrastive_ncm import (
    ContrastiveNCMDetector,
    DriftAutoencoder,
    NCMClassifier,
    train_plain_autoencoder,
)

DATA_DIR    = os.path.abspath('../../data/CICIDS2017_Engelen')
EPOCHS      = 300
LR          = 0.0001
BATCH_SIZE  = 4096
TEST_SIZE   = 0.25
N_RUNS      = 5
N_DROP      = 2
RANDOM_SEED = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.set_float32_matmul_precision('high')
print(f'Device: {DEVICE}')

Device: cuda


## 2. Load & Preprocess

In [2]:
DAY_FILES = [
    'Monday-WorkingHours.csv',
    'Tuesday-WorkingHours.csv',
    'Wednesday-WorkingHours.csv',
    'Thursday-WorkingHours.csv',
    'Friday-WorkingHours.csv',
]

print('Loading data...')
df = pd.concat(
    [pd.read_csv(os.path.join(DATA_DIR, f)) for f in DAY_FILES],
    ignore_index=True,
)
print(f'  Total rows: {len(df):,}')

df.drop(columns=['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Timestamp'],
        inplace=True, errors='ignore')
df['Label'] = df['Label'].apply(lambda x: 'BENIGN' if x.endswith('- Attempted') else x)
df['Label'] = df['Label'].replace({
    'DoS Hulk': 'DoS', 'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS', 'DoS Slowhttptest': 'DoS',
    'Web Attack - Brute Force': 'Web Attack', 'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
    'FTP-Patator': 'Patator', 'SSH-Patator': 'Patator',
})

X_raw = df[[c for c in df.columns if c != 'Label']].copy()
y_str = df['Label'].values
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
X_raw.dropna(axis=1, how='all', inplace=True)
X_raw.fillna(X_raw.mean(), inplace=True)

X       = StandardScaler().fit_transform(X_raw).astype(np.float32)
le_full = LabelEncoder()
y_all   = le_full.fit_transform(y_str)
input_dim = X.shape[1]
print(f'Features: {input_dim}  |  Classes: {list(le_full.classes_)}')

Loading data...
  Total rows: 2,100,814
Features: 77  |  Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS', 'Heartbleed', 'Infiltration', 'Patator', 'PortScan', 'Web Attack']


## 3. Helpers

In [ ]:
class GPUDataLoader:
    def __init__(self, X: torch.Tensor, y: torch.Tensor, batch_size: int, shuffle: bool = True):
        self.X, self.y, self.batch_size, self.n = X, y, batch_size, X.shape[0]
        self.shuffle = shuffle

    def __len__(self) -> int:
        return (self.n + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        idx = (torch.randperm(self.n, device=self.X.device)
               if self.shuffle else torch.arange(self.n, device=self.X.device))
        for i in range(0, self.n, self.batch_size):
            b = idx[i:i + self.batch_size]
            yield self.X[b], self.y[b]


def make_loader(X_arr, y_arr, shuffle=True):
    return GPUDataLoader(
        torch.from_numpy(X_arr).to(DEVICE),
        torch.from_numpy(y_arr).to(DEVICE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


def best_threshold(cal_dists, val_dists, val_labels, n_steps=300):
    best_f1, best_t, best_q = 0.0, 0.0, 0.0
    for q in np.linspace(0.01, 0.99, n_steps):
        t     = float(torch.quantile(cal_dists, q).item())
        preds = (val_dists > t).numpy().astype(int)
        f1    = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t, best_q = f1, t, q
    return best_t, best_q * 100


def evaluate(dists, y_true, threshold):
    preds = (dists > threshold).numpy().astype(int)
    return (
        precision_score(y_true, preds, zero_division=0),
        recall_score(y_true, preds, zero_division=0),
        f1_score(y_true, preds, zero_division=0),
    )


def run_experiment(dropped_classes):
    d_ids = np.array([le_full.transform([c])[0] for c in dropped_classes])

    X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
        X, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
    )
    mask_r      = ~np.isin(y_tr_r, d_ids)
    le_r        = LabelEncoder()
    y_tr_enc    = le_r.fit_transform(y_tr_r[mask_r]).astype(np.int64)
    n_classes_r = len(le_r.classes_)

    X_fit_r, X_cal_r, y_fit_r, _ = train_test_split(
        X_tr_r[mask_r], y_tr_enc, test_size=0.10,
        random_state=RANDOM_SEED, stratify=y_tr_enc,
    )

    rng_r   = np.random.default_rng(RANDOM_SEED)
    pos_idx = np.where(np.isin(y_te_r, d_ids))[0]
    neg_idx = np.where(~np.isin(y_te_r, d_ids))[0]
    bal_idx = np.concatenate([pos_idx, rng_r.choice(neg_idx, size=len(pos_idx), replace=False)])
    y_bin_r = np.isin(y_te_r[bal_idx], d_ids).astype(int)

    val_r, held_r = train_test_split(
        np.arange(len(bal_idx)), test_size=0.5, random_state=RANDOM_SEED, stratify=y_bin_r
    )

    X_val_r  = torch.from_numpy(X_te_r[bal_idx][val_r])
    y_val_r  = y_bin_r[val_r]
    X_held_r = torch.from_numpy(X_te_r[bal_idx][held_r])
    y_held_r = y_bin_r[held_r]
    X_cal_r  = torch.from_numpy(X_cal_r)

    loader_r = make_loader(X_fit_r, y_fit_r)
    det_r    = ContrastiveNCMDetector(input_dim=input_dim, hidden_dim=64, latent_dim=32,
                                      concept_threshold=3.5, device=DEVICE)
    ae_r     = DriftAutoencoder(input_dim=input_dim, hidden_dim=64, latent_dim=32).to(DEVICE)

    if DEVICE.type == 'cuda':
        stream_c, stream_p = torch.cuda.Stream(), torch.cuda.Stream()
        def _c():
            with torch.cuda.stream(stream_c):
                det_r.fit(loader_r, epochs=EPOCHS, lr=LR, num_classes=n_classes_r, tqdm_position=0)
        def _p():
            with torch.cuda.stream(stream_p):
                train_plain_autoencoder(ae_r, loader_r, epochs=EPOCHS, lr=LR, tqdm_position=1)
        with ThreadPoolExecutor(max_workers=2) as ex:
            fc, fp = ex.submit(_c), ex.submit(_p)
            fc.result(); fp.result()
        torch.cuda.synchronize()
    else:
        det_r.fit(loader_r, epochs=EPOCHS, lr=LR, num_classes=n_classes_r, tqdm_position=0)
        train_plain_autoencoder(ae_r, loader_r, epochs=EPOCHS, lr=LR, tqdm_position=1)

    ae_r.eval()
    all_h, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader_r:
            all_h.append(ae_r.encode(xb))
            all_y.append(yb)
    ncm_r = NCMClassifier(lambda_1=0.1)
    ncm_r.fit(torch.cat(all_h), torch.cat(all_y), n_classes_r)

    with torch.no_grad():
        h_held_e = ae_r.encode(X_held_r.to(DEVICE)).cpu()
    ed_held = torch.cdist(h_held_e, ncm_r.prototypes.cpu()).min(dim=1).values

    _, _, cd_c = det_r.detect(X_cal_r)
    _, _, vd_c = det_r.detect(X_val_r)
    _, _, td_c = det_r.detect(X_held_r)
    with torch.no_grad():
        cd_p = ncm_r._compute_distance(ae_r.encode(X_cal_r.to(DEVICE))).min(dim=1).values.cpu()
        vd_p = ncm_r._compute_distance(ae_r.encode(X_val_r.to(DEVICE))).min(dim=1).values.cpu()
        td_p = ncm_r._compute_distance(ae_r.encode(X_held_r.to(DEVICE))).min(dim=1).values.cpu()

    thresh_c, pct_c = best_threshold(cd_c, vd_c, y_val_r)
    thresh_p, pct_p = best_threshold(cd_p, vd_p, y_val_r)

    precision_c, recall_c, f1_c = evaluate(td_c, y_held_r, thresh_c)
    precision_p, recall_p, f1_p = evaluate(td_p, y_held_r, thresh_p)
        
    print(f'  ContrastiveNCM  precision={precision_c:.3f}  recall_c={recall_c:.3f}  f1={f1_c:.3f}   (thresh@{pct_c:.0f}%)')
    print(f'  Plain AE NCM    precision={precision_p:.3f}  recall_p={recall_p:.3f}  f1={f1_p:.3f}   (thresh@{pct_p:.0f}%)')

    return {
        'ContrastiveNCM': (precision_c, recall_c, f1_c, pct_c),
        'Plain AE NCM':   (precision_p, recall_p, f1_p, pct_p),
        '_auroc': {
            'ContrastiveNCM': roc_auc_score(y_held_r, td_c.numpy()) * 100,
            'Plain AE NCM':   roc_auc_score(y_held_r, td_p.numpy()) * 100,
            'Euclidean':      roc_auc_score(y_held_r, ed_held.numpy()) * 100,
        },
    }

## 4. Experiment

In [ ]:
ATTACK_CLASSES = [c for c in le_full.classes_ if c != 'BENIGN']

rng_runs = np.random.default_rng(0)
run_dropped = [
    list(rng_runs.choice(ATTACK_CLASSES, size=N_DROP, replace=False))
    for _ in range(N_RUNS)
]

print('Run  Dropped classes')
for i, dropped in enumerate(run_dropped):
    print(f'  {i+1}  {dropped}')

all_results = []
for i, dropped in enumerate(run_dropped):
    print(f'\n→ Run {i+1}/{N_RUNS}  dropped={dropped}')
    all_results.append(run_experiment(dropped))
    print(f'  ✓ done')

prf_methods = [k for k in all_results[0] if not k.startswith('_')]
print('\n' + '=' * 70)
print(f"{'Method':<22} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Thresh%':>9}")
print('=' * 70)
for method in prf_methods:
    ps   = np.mean([r[method][0] for r in all_results])
    rs   = np.mean([r[method][1] for r in all_results])
    fs   = np.mean([r[method][2] for r in all_results])
    pcts = np.mean([r[method][3] for r in all_results])
    print(f'{method:<22} {ps:>10.3f} {rs:>10.3f} {fs:>10.3f} {pcts:>8.1f}%')
print('=' * 70)

print('\n' + '=' * 45)
print(f"{'Method':<20} {'AUROC':>8}   {'Paper':>6}")
paper_auroc = {'ContrastiveNCM': 89.8, 'Euclidean': 52.3, 'Plain AE NCM': None}
print('=' * 45)
for method in ['ContrastiveNCM', 'Plain AE NCM', 'Euclidean']:
    vals  = np.mean([r['_auroc'][method] for r in all_results])
    paper = f"{paper_auroc[method]:.1f}" if paper_auroc[method] else '  —'
    print(f'{method:<20} {vals:>8.1f}   {paper:>6}')
print('=' * 45)

Run  Dropped classes
  1  [np.str_('Patator'), np.str_('Web Attack')]
  2  [np.str_('DoS'), np.str_('DDoS')]
  3  [np.str_('Web Attack'), np.str_('Bot')]
  4  [np.str_('Patator'), np.str_('Web Attack')]
  5  [np.str_('Heartbleed'), np.str_('Infiltration')]

→ Run 1/5  dropped=[np.str_('Patator'), np.str_('Web Attack')]


Plain AE:   0%|          | 0/300 [00:00<?, ?it/s]

Contrastive:   0%|          | 0/300 [00:00<?, ?it/s]

  ContrastiveNCM  precision=0.708  recall_c=0.988  f1=0.825   (thresh@57%)
  Plain AE NCM    precision=0.589  recall_p=0.983  f1=0.736   (thresh@33%)
  ✓ done

→ Run 2/5  dropped=[np.str_('DoS'), np.str_('DDoS')]


Contrastive:   0%|          | 0/300 [00:00<?, ?it/s]

...

Plain AE:   0%|          | 0/300 [00:00<?, ?it/s]

  ContrastiveNCM  precision=0.725  recall_c=0.984  f1=0.835   (thresh@62%)
  Plain AE NCM    precision=0.584  recall_p=0.982  f1=0.732   (thresh@31%)
  ✓ done

→ Run 5/5  dropped=[np.str_('Heartbleed'), np.str_('Infiltration')]


Plain AE:   0%|          | 0/300 [00:00<?, ?it/s]

Contrastive:   0%|          | 0/300 [00:00<?, ?it/s]

  ContrastiveNCM  precision=0.833  recall_c=1.000  f1=0.909   (thresh@84%)
  Plain AE NCM    precision=0.625  recall_p=1.000  f1=0.769   (thresh@63%)
  ✓ done

Method                  Precision     Recall         F1   Thresh%
ContrastiveNCM              0.804      0.975      0.879     75.8%
Plain AE NCM                0.597      0.986      0.744     37.8%

Method                  AUROC    Paper
ContrastiveNCM           84.5     89.8
Plain AE NCM             57.1        —
Euclidean                75.9     52.3
